# Notebook 06: All Figures for Paper

Generates every figure in the Knowledge Organization paper from the
appendix CSV files produced by earlier notebooks.

**Run this notebook last**, after notebooks 01–05 have completed and
all appendix CSVs are populated.

Figures produced:
- `fig_rq1_registry_quality.pdf`    — registry composition (from notebook 02)
- `fig_pr_curves.pdf`               — precision-recall curves (from notebook 05)
- `fig_perplexity_distribution.pdf` — PLL distributions (from notebook 04)
- `fig_rq3_registry_growth.pdf`     — registry vs. retractions (RQ3)
- `fig_ablation_f1.pdf`             — ablation F1 bar chart (from notebook 05)

All figures are saved to `figures/` as 300 DPI PDFs ready for journal submission.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib
matplotlib.rcParams.update({
    'figure.dpi':      150,
    'font.size':       11,
    'axes.titlesize':  12,
    'axes.labelsize':  11,
    'legend.fontsize': 10,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
})

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

APPENDIX_DIR = ROOT / 'data' / 'appendix'
FIGURES_DIR  = ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

# Colour palette (consistent across all figures)
C_BLUE    = '#2166ac'
C_RED     = '#d73027'
C_ORANGE  = '#f46d43'
C_DARK    = '#053061'
C_LIGHT   = '#4393c3'

print('Figure output directory:', FIGURES_DIR)
print('Appendix data directory:', APPENDIX_DIR)

## Figure 1 (RQ1): Registry quality overview

In [ ]:
reg_path = APPENDIX_DIR / 'appendix_a_registry_export.csv'
if not reg_path.exists():
    print('Run notebook 02 first to generate appendix_a_registry_export.csv')
else:
    df_reg = pd.read_csv(reg_path)
    df_conf = df_reg[df_reg['status'] == 'confirmed']

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # (a) Domain distribution
    domain_counts = df_conf['domain'].value_counts()
    axes[0].barh(domain_counts.index, domain_counts.values, color=C_BLUE)
    axes[0].set_xlabel('Confirmed signals')
    axes[0].set_title('(a) Registry composition by domain')

    # (b) Precision distribution
    prec = df_conf['precision_on_clean'].dropna()
    axes[1].hist(prec, bins=8, color=C_BLUE, edgecolor='white',
                 range=(0.90, 1.01))
    axes[1].axvline(x=0.95, color=C_RED, linestyle='--', linewidth=2,
                   label='CI gate threshold (0.95)')
    axes[1].set_xlabel('Precision on clean corpus')
    axes[1].set_title('(b) Precision distribution (confirmed signals)')
    axes[1].legend()

    # (c) Warrant gate pass rates
    lit_rate  = df_reg['warrant_literary'].mean()
    ter_rate  = df_reg['warrant_terminological'].mean()
    stat_rate = df_reg['warrant_statistical'].mean()
    labels    = ['Literary', 'Terminological', 'Statistical']
    rates     = [lit_rate, ter_rate, stat_rate]
    colors    = [C_LIGHT, C_BLUE, C_DARK]
    bars = axes[2].bar(labels, [r * 100 for r in rates], color=colors)
    axes[2].set_ylabel('Pass rate (%)')
    axes[2].set_ylim(0, 115)
    axes[2].set_title('(c) Warrant gate pass rates (all signals)')
    for bar, rate in zip(bars, rates):
        axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{rate:.0%}', ha='center', fontsize=11)

    plt.suptitle(
        'Figure 1: Warrant-based signal registry quality (RQ1)\n'
        'Three-gate governance produces a high-precision, expert-validated vocabulary of terminological violations',
        y=1.03, fontsize=11
    )
    plt.tight_layout()
    out = FIGURES_DIR / 'fig_rq1_registry_quality.pdf'
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## Figure 2 (RQ2): Ablation study F1 scores

In [ ]:
abl_path = APPENDIX_DIR / 'appendix_c_ablation_results.csv'
if not abl_path.exists():
    print('Run notebook 05 first to generate appendix_c_ablation_results.csv')
else:
    df_abl = pd.read_csv(abl_path)
    # Focus on positive+negative corpus
    df_pn = df_abl[df_abl['corpus'] == 'positive+negative'].copy()

    if df_pn.empty:
        print('No positive+negative corpus rows — using all corpus rows.')
        df_pn = df_abl.copy()

    df_pn = df_pn.sort_values('f1')

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # (a) F1 bar chart by combination
    colours = [C_BLUE if c != 'L1+L2+L3' else C_DARK for c in df_pn['combination']]
    bars = axes[0].barh(df_pn['combination'], df_pn['f1'], color=colours)
    axes[0].set_xlabel('F1 Score')
    axes[0].set_xlim(0, 1.05)
    axes[0].set_title('(a) F1 per layer combination')
    for bar, val in zip(bars, df_pn['f1']):
        if val > 0:
            axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                         f'{val:.3f}', va='center', fontsize=9)

    # (b) Precision vs Recall scatter
    for _, row in df_pn.iterrows():
        color = C_DARK if row['combination'] == 'L1+L2+L3' else C_BLUE
        size  = 120 if row['combination'] == 'L1+L2+L3' else 60
        axes[1].scatter(row['recall'], row['precision'],
                        c=color, s=size, zorder=5)
        axes[1].annotate(
            row['combination'],
            (row['recall'], row['precision']),
            fontsize=8, xytext=(5, 3), textcoords='offset points'
        )
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title('(b) Precision vs. Recall per combination')
    axes[1].axhline(y=0.95, color=C_RED, linestyle='--', alpha=0.5, label='Precision gate (0.95)')
    axes[1].legend()

    plt.suptitle(
        'Figure 2: Ablation study — layer combinations on positive+negative corpus (RQ2)\n'
        'Combined L1+L2+L3 achieves best F1; L1 maintains near-perfect precision',
        y=1.03, fontsize=11
    )
    plt.tight_layout()
    out = FIGURES_DIR / 'fig_ablation.pdf'
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## Figure 3 (RQ2): Perplexity distributions

In [ ]:
# This figure is generated by notebook 04 directly.
# If notebook 04 has been run, the figure already exists at:
fig3_path = FIGURES_DIR / 'fig_perplexity_distribution.pdf'
if fig3_path.exists():
    print(f'Figure 3 already exists: {fig3_path}')
    print('Generated by notebook 04 (Layer 3 MLM perplexity evaluation).')
else:
    print('Figure 3 not found. Run notebook 04 to generate it.')

## Figure 4 (RQ3): Registry growth vs. retraction timeline

In [ ]:
from tpc.evaluation.registry_growth import (
    plot_registry_vs_retractions,
    export_appendix_e
)

print('Generating Figure 4 (RQ3 registry growth)...')
print('Note: requires internet access for Retraction Watch API.')
print('Falls back to synthetic timeline if API is unavailable.')

combined = plot_registry_vs_retractions(
    signals_dir=ROOT / 'signals' / 'phrases',
    output_path=str(FIGURES_DIR / 'fig_rq3_registry_growth.pdf'),
    show=True
)

# Export Appendix E
export_appendix_e(
    combined,
    output_path=str(APPENDIX_DIR / 'appendix_e_registry_growth.csv')
)

print('\n=== Table 7 values (copy into Section 6.3) ===')
print(f'Pearson r (lag=0):        {combined.attrs.get("pearson_r_lag0", "N/A")}')
print(f'Pearson r (lag=3 months): {combined.attrs.get("pearson_r_lag3", "N/A")}')
print(f'Appendix E saved: data/appendix/appendix_e_registry_growth.csv')

## Figure 5 (RQ3): Domain and tool-origin trajectories

In [ ]:
import yaml
from tpc.evaluation.registry_growth import load_registry_timeline

reg_df = load_registry_timeline(ROOT / 'signals' / 'phrases')

if reg_df['discovery_date'].notna().sum() < 2:
    print('Not enough dated signals for trajectory analysis.')
    print('Discovery dates will be populated as the real registry grows.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # (a) Registry growth by domain
    for domain, grp in reg_df.groupby('domain'):
        monthly = (grp.set_index('discovery_date')
                      .resample('ME').size()
                      .cumsum())
        if len(monthly) > 1:
            axes[0].plot(monthly.index, monthly.values, marker='o',
                         markersize=4, label=domain)
    axes[0].set_xlabel('Date')
    axes[0].set_ylabel('Cumulative confirmed signals')
    axes[0].set_title('(a) Registry growth by domain')
    axes[0].legend()

    # (b) Registry growth by paraphrase tool origin
    for tool, grp in reg_df.groupby('paraphrase_tool_origin'):
        if pd.isna(tool):
            continue
        monthly = (grp.set_index('discovery_date')
                      .resample('ME').size()
                      .cumsum())
        if len(monthly) > 1:
            axes[1].plot(monthly.index, monthly.values, marker='s',
                         markersize=4, label=str(tool)[:30])
    axes[1].set_xlabel('Date')
    axes[1].set_ylabel('Cumulative confirmed signals')
    axes[1].set_title('(b) Registry growth by paraphrase tool')
    axes[1].legend(fontsize=8)

    plt.suptitle(
        'Figure 5: Domain and tool-origin trajectories (RQ3)\n'
        'Registry growth stratified by scientific domain and paraphrase tool origin',
        y=1.03, fontsize=11
    )
    plt.tight_layout()
    out = FIGURES_DIR / 'fig_rq3_trajectories.pdf'
    plt.savefig(out, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out}')

## Final summary: all figures generated

In [ ]:
print('=== PAPER FIGURE INVENTORY ===')
figure_map = {
    'fig_rq1_registry_quality.pdf':     'Figure 1 — Registry quality (RQ1)',
    'fig_ablation.pdf':                 'Figure 2 — Ablation study (RQ2)',
    'fig_perplexity_distribution.pdf':  'Figure 3 — PLL distributions (RQ2)',
    'fig_rq3_registry_growth.pdf':      'Figure 4 — Registry growth vs. retractions (RQ3)',
    'fig_rq3_trajectories.pdf':         'Figure 5 — Domain and tool trajectories (RQ3)',
}
for filename, label in figure_map.items():
    path   = FIGURES_DIR / filename
    status = '✓ EXISTS' if path.exists() else '✗ MISSING (run relevant notebook)'
    print(f'  {label}')
    print(f'    {filename}: {status}')

print()
print('=== APPENDIX CSV INVENTORY ===')
appendix_map = {
    'appendix_a_registry_export.csv':   'Appendix A — Registry export (RQ1) — notebook 02',
    'appendix_b_corpus_statistics.csv': 'Appendix B — Corpus statistics   — notebook 01',
    'appendix_c_ablation_results.csv':  'Appendix C — Ablation results    — notebook 05',
    'appendix_d_novel_candidates.csv':  'Appendix D — Novel candidates     — notebook 04',
    'appendix_e_registry_growth.csv':   'Appendix E — Growth timeline      — this notebook',
}
for filename, label in appendix_map.items():
    path   = APPENDIX_DIR / filename
    if path.exists():
        df = pd.read_csv(path)
        status = f'✓ EXISTS ({len(df)} rows)'
    else:
        status = '✗ MISSING'
    print(f'  {label}')
    print(f'    {filename}: {status}')